# 🔬 Notebook 02 — Pipeline Visualisation
See the output of every preprocessing step on a real image.
Use this to tune parameters in config.py.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
import importlib
import sys; sys.path.insert(0, '..')
# from preprocessing.config import DATASET_RAW, IMG_EXTENSIONS
from preprocessing.shared.base import (
    check_quality, step1_resize, step2_remove_background,
    step3_histogram_matching, step4_bilateral_filter
)
from preprocessing.species_id.transforms_sid import step5a_clahe, step6a_sharpen
import features.species_id.leaflet_segmentation as leaflet_segmentation
importlib.reload(leaflet_segmentation)
segment_leaflets = leaflet_segmentation.segment_leaflets
print('✅ All imports ok (leaflet_segmentation reloaded)')

✅ All imports ok


## Pick a test image
Change `TEST_IMAGE` to any image in your dataset.

In [21]:
# ── Change this to any image path ──────────────────────────────────────────
IMG_EXTENSIONS      = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
DATASET_RAW = '../dataset/raw/kasthuri_dehi/'
raw = Path(DATASET_RAW)
all_imgs = list(raw.rglob('*.jpg')) + list(raw.rglob('*.png'))

if not all_imgs:
    print('⚠️  No images found. Add images to dataset/raw/species_name/')
else:
    TEST_IMAGE = all_imgs[0]   # ← change index or set a specific path
    print(f'Using: {TEST_IMAGE}')
    raw_img = cv2.imread(str(TEST_IMAGE))
    print(f'Original shape: {raw_img.shape}')

Using: ..\dataset\raw\kasthuri_dehi\PXL_20260429_050545860.jpg
Original shape: (4032, 3024, 3)


## Run each step and show result

In [25]:
if not all_imgs:
    print('No images to process')
else:
    ok, blur = check_quality(raw_img)
    print(f'Blur score: {blur}  →  {"PASS ✅" if ok else "REJECT ❌"}')

    s1 = step1_resize(raw_img)
    s2_img, s2_mask = step2_remove_background(s1)
    s3 = step3_histogram_matching(s2_img)
    s4 = step4_bilateral_filter(s3)
    s5 = step5a_clahe(s4)
    s6 = step6a_sharpen(s5)
    leaflets, label_vis = segment_leaflets(s6, s2_mask)

    stages = [
        (raw_img, f'0 · Original\n{raw_img.shape[1]}×{raw_img.shape[0]}'),
        (s1,      f'1 · Resized\n512×512'),
        (s2_img,  '2 · BG Removed\n(Saliency+GrabCut)'),
        (s2_mask, '2 · Binary Mask'),
        (s4,      '4 · Bilateral Filter'),
        (s5,      '5a · CLAHE'),
        (s6,      '6a · Sharpened'),
        (label_vis,f'7a · Leaflets\ncount={len(leaflets)}'),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f'Pipeline Stages — {TEST_IMAGE.name}', fontsize=13, fontweight='bold')
    for ax, (img_s, title) in zip(axes.flat, stages):
        if len(img_s.shape) == 2:
            ax.imshow(img_s, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img_s, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()
    print(f'Leaflets found: {len(leaflets)}')

Blur score: 43.56  →  REJECT ❌


NameError: name 'area' is not defined

## Run on multiple images — compare species side by side

In [ ]:
from preprocessing.species_id.pipeline_sid import preprocess_one

raw = Path(DATASET_RAW)
species_dirs = [d for d in raw.iterdir() if d.is_dir()]

sample_imgs = []
for sp_dir in species_dirs[:5]:   # max 5 species for readability
    imgs = list(sp_dir.glob('*.jpg')) + list(sp_dir.glob('*.png'))
    if imgs:
        sample_imgs.append((sp_dir.name, imgs[0]))

if not sample_imgs:
    print('No images found')
else:
    fig, axes = plt.subplots(len(sample_imgs), 3, figsize=(12, 4*len(sample_imgs)))
    if len(sample_imgs) == 1: axes = [axes]
    fig.suptitle('Before / After / Mask — per Species', fontsize=12, fontweight='bold')

    for (sp_name, img_path), row in zip(sample_imgs, axes):
        raw_img = cv2.imread(str(img_path))
        result, mask, meta = preprocess_one(img_path)

        row[0].imshow(cv2.cvtColor(cv2.resize(raw_img,(256,256)), cv2.COLOR_BGR2RGB))
        row[0].set_title(f'{sp_name}\nOriginal', fontsize=9)

        if result is not None:
            row[1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            row[1].set_title(f'Preprocessed\nblur={meta["blur_score"]}', fontsize=9)
            row[2].imshow(mask, cmap='gray')
            row[2].set_title('Leaf Mask', fontsize=9)
        else:
            row[1].text(0.5,0.5, f'REJECTED\n{meta["reason"]}',
                       ha='center', va='center', transform=row[1].transAxes, color='red')
            row[2].axis('off')

        for ax in row: ax.axis('off')

    plt.tight_layout(); plt.show()

## Tune: adjust BLUR_THRESHOLD in config.py if too many rejected

In [ ]:
# Quick threshold sensitivity check
from preprocessing.config import BLUR_THRESHOLD
raw = Path(DATASET_RAW)
scores = []
for p in raw.rglob('*'):
    if p.suffix.lower() not in IMG_EXTENSIONS: continue
    img = cv2.imread(str(p))
    if img is not None:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        scores.append(cv2.Laplacian(gray, cv2.CV_64F).var())

for thresh in [50, 80, 100, 120]:
    rejected = sum(1 for s in scores if s < thresh)
    pct = 100*rejected/max(len(scores),1)
    marker = '← current' if thresh == BLUR_THRESHOLD else ''
    print(f'  threshold={thresh:4d}  rejects {rejected:3d}/{len(scores)} images ({pct:.1f}%)  {marker}')